# Supply and Demand Zones from OHLCV

This notebook shows a simple, explainable way to label supply and demand zones from OHLCV candles.

- **Demand zone**: tight base followed by a strong rally.
- **Supply zone**: tight base followed by a strong drop.

The logic is intentionally simple so it can later become a derivative/feature in the miner.

In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams['figure.figsize'] = (18, 9)
plt.rcParams['axes.grid'] = True

## 1. Load Sample Stock Data

Change `SYMBOL` if you want to inspect another stock. The default uses the archived derivative cache.

In [ ]:
SYMBOL = 'KERNEX'
DATA_DIR = r'E:/archieve/july_2026/miner_signal_notebook/derivative_cache'

def find_symbol_csv(symbol, data_dir):
    symbol = symbol.upper().replace('.NS', '')
    candidates = []
    for name in os.listdir(data_dir):
        upper = name.upper()
        if upper == f'{symbol}.CSV' or upper.startswith(f'{symbol}.NS_') or upper.startswith(f'{symbol}_'):
            candidates.append(os.path.join(data_dir, name))
    if not candidates:
        raise FileNotFoundError(f'No CSV found for {symbol} in {data_dir}')
    return sorted(candidates)[0]

csv_path = find_symbol_csv(SYMBOL, DATA_DIR)
df = pd.read_csv(csv_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Keep only required OHLCV columns.
df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
df[['Open', 'High', 'Low', 'Close', 'Volume']] = df[['Open', 'High', 'Low', 'Close', 'Volume']].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=['Open', 'High', 'Low', 'Close']).reset_index(drop=True)

print(csv_path)
print(df.head())
print(df.tail())

## 2. Define Zone Detection Logic

For each possible base:

1. Check whether the base is tight/narrow.
2. Check whether price leaves the base strongly.
3. If it rallies, create a demand zone.
4. If it drops, create a supply zone.

This is not a magic formula. It is a starting point we can tune.

In [ ]:
def add_atr(data, period=14):
    data = data.copy()
    prev_close = data['Close'].shift(1)
    tr1 = data['High'] - data['Low']
    tr2 = (data['High'] - prev_close).abs()
    tr3 = (data['Low'] - prev_close).abs()
    data['TR'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    data['ATR'] = data['TR'].rolling(period, min_periods=period).mean()
    return data

def detect_supply_demand_zones(
    data,
    min_base_candles=2,
    max_base_candles=6,
    base_atr_multiplier=1.20,
    move_lookahead=5,
    move_atr_multiplier=1.80,
    min_move_pct=3.0,
):
    zones = []
    data = add_atr(data)
    n = len(data)

    for start in range(20, n - max_base_candles - move_lookahead):
        for base_len in range(min_base_candles, max_base_candles + 1):
            end = start + base_len - 1
            out_end = end + move_lookahead
            base = data.iloc[start:end + 1]
            after = data.iloc[end + 1:out_end + 1]
            atr = data.loc[end, 'ATR']

            if not np.isfinite(atr) or atr <= 0:
                continue

            base_high = base['High'].max()
            base_low = base['Low'].min()
            base_range = base_high - base_low
            base_mid = (base_high + base_low) / 2
            base_range_pct = (base_range / base_mid) * 100 if base_mid else np.nan

            # Tight base: range should be small relative to ATR.
            if base_range > atr * base_atr_multiplier:
                continue

            base_close = data.loc[end, 'Close']
            max_after_high = after['High'].max()
            min_after_low = after['Low'].min()
            rally_move = max_after_high - base_close
            drop_move = base_close - min_after_low
            rally_pct = (rally_move / base_close) * 100 if base_close else 0
            drop_pct = (drop_move / base_close) * 100 if base_close else 0

            # Demand zone: tight base followed by strong rally.
            if rally_move >= atr * move_atr_multiplier and rally_pct >= min_move_pct:
                body_high = base[['Open', 'Close']].max(axis=1).max()
                zones.append({
                    'type': 'demand',
                    'start_idx': start,
                    'end_idx': end,
                    'created_idx': end + 1,
                    'zone_low': base_low,
                    'zone_high': body_high,
                    'move_pct': rally_pct,
                    'base_range_pct': base_range_pct,
                    'score': rally_pct / max(base_range_pct, 0.01),
                })

            # Supply zone: tight base followed by strong drop.
            if drop_move >= atr * move_atr_multiplier and drop_pct >= min_move_pct:
                body_low = base[['Open', 'Close']].min(axis=1).min()
                zones.append({
                    'type': 'supply',
                    'start_idx': start,
                    'end_idx': end,
                    'created_idx': end + 1,
                    'zone_low': body_low,
                    'zone_high': base_high,
                    'move_pct': drop_pct,
                    'base_range_pct': base_range_pct,
                    'score': drop_pct / max(base_range_pct, 0.01),
                })

    zones_df = pd.DataFrame(zones)
    if len(zones_df) > 0:
        zones_df['start_date'] = zones_df['start_idx'].map(data['Date'])
        zones_df['end_date'] = zones_df['end_idx'].map(data['Date'])
        zones_df = zones_df.sort_values('score', ascending=False).reset_index(drop=True)
    return data, zones_df

df2, zones = detect_supply_demand_zones(df)
print('zones:', len(zones))
zones.head(20)

## 3. Plot Candles with Zones

Green rectangles are demand zones. Red rectangles are supply zones.

In [ ]:
def plot_candles_with_zones(data, zones_df, lookback=260, max_zones=20):
    plot_data = data.tail(lookback).copy().reset_index(drop=False).rename(columns={'index': 'original_idx'})
    min_original_idx = int(plot_data['original_idx'].min())
    max_original_idx = int(plot_data['original_idx'].max())

    active_zones = zones_df[
        (zones_df['created_idx'] <= max_original_idx) &
        (zones_df['end_idx'] >= min_original_idx - 260)
    ].head(max_zones).copy()

    fig, ax = plt.subplots()
    width = 0.65

    # Draw zones first.
    for _, z in active_zones.iterrows():
        x0 = max(0, int(z['created_idx'] - min_original_idx))
        x1 = len(plot_data) - 1
        color = 'green' if z['type'] == 'demand' else 'red'
        alpha = 0.14 if z['type'] == 'demand' else 0.12
        ax.add_patch(Rectangle(
            (x0, z['zone_low']),
            max(1, x1 - x0),
            z['zone_high'] - z['zone_low'],
            color=color,
            alpha=alpha,
            linewidth=0,
        ))
        ax.text(x0, z['zone_high'], f"{z['type']} {z['score']:.1f}", color=color, fontsize=8, va='bottom')

    # Draw candles.
    for i, row in plot_data.iterrows():
        o, h, l, c = row['Open'], row['High'], row['Low'], row['Close']
        color = '#16a34a' if c >= o else '#dc2626'
        ax.plot([i, i], [l, h], color=color, linewidth=1)
        body_low = min(o, c)
        body_height = max(abs(c - o), 0.001)
        ax.add_patch(Rectangle((i - width/2, body_low), width, body_height, color=color, alpha=0.85))

    tick_step = max(1, len(plot_data) // 10)
    ticks = list(range(0, len(plot_data), tick_step))
    labels = [plot_data.loc[t, 'Date'].strftime('%Y-%m-%d') for t in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title(f'{SYMBOL} Supply/Demand Zones - Last {lookback} Candles')
    ax.set_ylabel('Price')
    plt.tight_layout()
    return fig, ax

plot_candles_with_zones(df2, zones, lookback=260, max_zones=20)
plt.show()

## 4. Convert Zones into Miner Features

Once zones are detected, we can label every candle with features like:

- `inside_demand_zone`
- `inside_supply_zone`
- `distance_to_nearest_demand_zone_pct`
- `distance_to_nearest_supply_zone_pct`
- `nearest_demand_zone_score`
- `nearest_supply_zone_score`

Those columns can then be added to the mining pipeline.

In [ ]:
def label_zone_features(data, zones_df):
    output = data.copy()
    output['inside_demand_zone'] = 0
    output['inside_supply_zone'] = 0
    output['distance_to_nearest_demand_zone_pct'] = np.nan
    output['distance_to_nearest_supply_zone_pct'] = np.nan
    output['nearest_demand_zone_score'] = np.nan
    output['nearest_supply_zone_score'] = np.nan

    for i, row in output.iterrows():
        close = row['Close']
        past_zones = zones_df[zones_df['created_idx'] <= i]

        for zone_type in ['demand', 'supply']:
            zdf = past_zones[past_zones['type'] == zone_type]
            if zdf.empty:
                continue

            distances = []
            for _, z in zdf.iterrows():
                if z['zone_low'] <= close <= z['zone_high']:
                    distance_pct = 0.0
                elif close < z['zone_low']:
                    distance_pct = ((z['zone_low'] - close) / close) * 100
                else:
                    distance_pct = ((close - z['zone_high']) / close) * 100
                distances.append((distance_pct, z))

            distances.sort(key=lambda x: x[0])
            nearest_distance, nearest_zone = distances[0]

            if zone_type == 'demand':
                output.loc[i, 'distance_to_nearest_demand_zone_pct'] = nearest_distance
                output.loc[i, 'nearest_demand_zone_score'] = nearest_zone['score']
                if nearest_distance == 0:
                    output.loc[i, 'inside_demand_zone'] = 1
            else:
                output.loc[i, 'distance_to_nearest_supply_zone_pct'] = nearest_distance
                output.loc[i, 'nearest_supply_zone_score'] = nearest_zone['score']
                if nearest_distance == 0:
                    output.loc[i, 'inside_supply_zone'] = 1

    return output

labeled = label_zone_features(df2, zones)
labeled[['Date', 'Close', 'inside_demand_zone', 'inside_supply_zone', 'distance_to_nearest_demand_zone_pct', 'distance_to_nearest_supply_zone_pct', 'nearest_demand_zone_score', 'nearest_supply_zone_score']].tail(20)